# The graph

**Two independent time axes, and traversal that runs in the database.** Blast radius over a ten-thousand-node graph is one recursive query, not ten thousand round trips.

> **Every cell in this notebook runs.** They are generated from
> [`tools/notebooks/spec.py`](../tools/notebooks/spec.py) and executed by CI, so a
> cell that cannot run does not reach a commit. Change a cell, re-run it, and the
> page is yours — that is what it is for.


Two things make this graph different from a diagram:

**It is bitemporal.** `valid_from`/`valid_to` says when something was true in the
world; `observed_at`/`superseded_at` says when we learned it. Those are different
questions and conflating them makes "what did we believe last Tuesday?"
unanswerable.

**Nothing is deleted.** A correction supersedes; a retirement sets `valid_to`.
History survives, which is what lets an audit ask what was known at the time a
decision was made.

In [ ]:
# --- setup: works locally, on Binder, and on Colab -------------------------
import subprocess, sys, pathlib

def _ensure_installed():
    """Make the package importable, preferring the checkout this notebook is in.

    The checkout comes first deliberately. Trusting whichever `slpie` happens to
    be installed means a notebook opened inside one clone can silently exercise
    a different one — which is exactly what happened while writing this.
    """
    here = pathlib.Path.cwd()
    root = next(
        (p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), None,
    )
    if root is None:                     # Colab: no checkout, so fetch one
        root = pathlib.Path("/content/Macropol-s")
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/Reimain/Macropol-s.git", str(root)],
                check=True,
            )
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    try:
        import slpie, gratimos          # noqa: F401
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)],
                       check=True)
    return root

ROOT = _ensure_installed()
print("package root:", ROOT)

import slpie
print("slpie", slpie.__version__)

## Build one

In [ ]:
import tempfile, pathlib
from slpie.graph.sqlite_graph import SqliteGraph
from slpie.graph.traversal import Traverser
from slpie.domain.node import Node, NodeKind
from slpie.domain.edge import Edge, EdgeKind
from slpie.domain.identity import Purl
from slpie.domain.evidence import Evidence, EvidenceKind, SourceLocation

WORK = pathlib.Path(tempfile.mkdtemp(prefix="slpie-nb-"))
graph = SqliteGraph(WORK / "graph.db")

def cite(name):
    return Evidence(
        kind=EvidenceKind.LOCKFILE_PIN,
        location=SourceLocation("file:///r/package-lock.json", line=1),
        extractor="npm", extractor_version="1", excerpt=f'"{name}"',
    )

def package(name, version="1.0.0"):
    return Node(kind=NodeKind.PACKAGE,
                identity=Purl.parse(f"pkg:npm/{name}@{version}"),
                evidence=(cite(name),))

# app -> api -> auth -> crypto, and app -> ui -> crypto
names = ["app", "api", "auth", "crypto", "ui"]
nodes = {name: package(name) for name in names}
graph.assert_nodes(nodes.values(), sequence=1)

chain = [("app", "api"), ("api", "auth"), ("auth", "crypto"),
         ("app", "ui"), ("ui", "crypto")]
graph.assert_edges(
    [Edge(kind=EdgeKind.DEPENDS_ON, src=nodes[a].id, dst=nodes[b].id,
          evidence=(cite(f"{a}->{b}"),)) for a, b in chain],
    sequence=1,
)

print("nodes:", graph.counts()["nodes"], "| edges:", graph.counts()["edges"])

## Blast radius

"If `crypto` changes, what breaks?" — reverse reachability, with a confidence floor and a cycle guard, executed as one SQL query.

In [ ]:
traverser = Traverser(graph)
impact = traverser.impact(nodes["crypto"].id, max_depth=10)

print(f"{len(impact.impacted)} node(s) depend on crypto, directly or otherwise\n")
for entry in impact.impacted:
    name = (entry.display or entry.node_id).split("/")[-1]
    print(f"  distance {entry.distance}  confidence {entry.confidence:.2f}  {name}")

`path_confidence` propagates the **minimum** along the path. A node reached only through a 0.4 dynamic load is reported as reached at 0.4 — so "this is affected" and "we think this might be affected" are not the same answer wearing the same face.

In [ ]:
# The same query forward: what does `app` depend on?
downstream = traverser.dependencies(nodes["app"].id, max_depth=10)
print(f"app reaches {len(downstream.impacted)} node(s):")
for entry in downstream.impacted:
    name = (entry.display or entry.node_id).split("/")[-1]
    print(f"  distance {entry.distance}  {name}")

## Cycles

Same recursive CTE, pointed at itself.

In [ ]:
# Introduce one: crypto -> app closes the loop.
graph.assert_edges(
    [Edge(kind=EdgeKind.DEPENDS_ON, src=nodes["crypto"].id, dst=nodes["app"].id,
          evidence=(cite("crypto->app"),))],
    sequence=2,
)

cycles = traverser.cycles(max_depth=10)
print(f"{len(cycles)} cycle(s) found")
for cycle in cycles[:3]:
    names_in = [str(name).split("/")[-1] for name in (cycle.displays or cycle.nodes)]
    print("  " + " → ".join(names_in))

## Nothing is deleted

Retire a node and it stops being live — but it is still there, and the history still answers.

In [ ]:
import time

before = graph.counts()
graph.retire_node(nodes["ui"].id, valid_to=time.time_ns(), sequence=3)
after = graph.counts()

print("live nodes before retirement:", before["nodes"])
print("live nodes after retirement: ", after["nodes"])
print()
print("The row is not gone — retirement sets `valid_to`, so history survives")
print("and a bitemporal query still answers what was believed before it.")

## Snapshots are content-addressed

Identical inputs produce an identical snapshot id, so "is this the same architecture as last release?" is a string comparison.

In [ ]:
from slpie.graph.snapshot import SnapshotStore

store = SnapshotStore(graph)
first = store.seal(ledger_version=3, label="baseline")
second = store.seal(ledger_version=3, label="again")

print("first: ", first.root_digest[:32])
print("second:", second.root_digest[:32])
print("identical inputs, identical digest:", first.root_digest == second.root_digest)

graph.close()

## Your turn

Change a confidence floor and watch the blast radius shrink:

```python
traverser.impact(nodes['crypto'].id, max_depth=10, min_confidence=0.95)
```

In [ ]:
# Scratch cell.
strict = traverser.impact(nodes["crypto"].id, max_depth=10, min_confidence=0.99)
print("at min_confidence=0.99:", len(strict.impacted), "node(s)")